# Package

In [11]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

# ----------------------------
# Nixtla MLForecast
# ----------------------------
from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals

# ----------------------------
# Model backend
# ----------------------------
from sklearn.linear_model import LinearRegression

# Importation des données

In [12]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
from pathlib import Path
import pandas as pd
from feast import FeatureStore

def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())

# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    # IMPORTANT: full_feature_names=True pour éviter collisions / noms stables
    return fs.get_historical_features(
        entity_df=entity_df,
        features=feature_refs,
        full_feature_names=True,
    ).to_df()

# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"

series_ids = [
    "BUSLOANS",
    "CPIAUCSL",
    "DPCERA3M086SBEA",
    "INDPRO",
    "M2SL",
    "OILPRICEX",
    "RPI",
    "SP500",
    "TB3MS",
    "UNRATE",
    "USREC",
]

FEATURE_REFS = ["stationary_value:value"]

# ----------------------------
# 1) Dates de référence via calendrier mensuel + filtrage implicite par Feast
# ----------------------------
calendar = pd.date_range(start=START, end=END, freq=FREQ)

# Option robuste: on "valide" les dates réellement présentes en prenant UNRATE
entity_df_unrate = pd.DataFrame({"series_id": ["UNRATE"] * len(calendar), "date": calendar})
df_unrate_dates = load_features_from_feast(entity_df_unrate, ["raw_value:value"])

dates = (
    pd.to_datetime(df_unrate_dates["date"], utc=True, errors="coerce")
      .dt.tz_convert(None)
      .dropna()
      .sort_values()
      .unique()
)

print("Nb dates:", len(dates))
print("Date min:", dates.min(), "| Date max:", dates.max())

# ----------------------------
# 2) Entity DF multi-séries × dates (long)
# ----------------------------
entity_df = (
    pd.MultiIndex.from_product([series_ids, dates], names=["series_id", "date"])
      .to_frame(index=False)
)

# ----------------------------
# 3) Fetch stationary features (long)
# ----------------------------
df_stationary = load_features_from_feast(entity_df, FEATURE_REFS)

# normaliser date
df_stationary["date"] = pd.to_datetime(df_stationary["date"], utc=True, errors="coerce").dt.tz_convert(None)

# colonne valeur stable
value_col = "stationary_value__value"
if value_col not in df_stationary.columns:
    candidates = [c for c in df_stationary.columns if c.endswith("__value")]
    raise KeyError(f"Expected '{value_col}' not found. Candidates: {candidates}")

df_stationary = df_stationary[["series_id", "date", value_col]].rename(columns={value_col: "value"})
print("df_stationary shape:", df_stationary.shape)
print(df_stationary.head())

# ----------------------------
# 4) Construire dataset régression (logique article: full-information)
#    y_t = UNRATE(t)
#    features = UNRATE(t-12) + X(t-12)
# ----------------------------
H = 12  # horizon = 12 mois (cohérent article)

# Target y(t)
df_y = (
    df_stationary[df_stationary["series_id"] == "UNRATE"]
    .sort_values("date")
    .rename(columns={"value": "y"})
    .reset_index(drop=True)
)

# y(t-12)
df_y["y_lag12"] = df_y["y"].shift(H)

# Exogènes en colonnes à la date t
df_x_long = df_stationary[df_stationary["series_id"] != "UNRATE"].copy()
df_x = (
    df_x_long
    .pivot_table(index="date", columns="series_id", values="value", aggfunc="last")
    .sort_index()
)

# ✅ Full-information: X(t-12) aligné sur la date t
df_x_lag = df_x.shift(H)

# (Optionnel mais recommandé) suffixe explicite
df_x_lag = df_x_lag.add_suffix("_lag12").reset_index()

# Merge final: y(t) avec y(t-12) + X(t-12)
df_model = (
    df_y[["date", "y", "y_lag12"]]
    .merge(df_x_lag, on="date", how="left")
    .dropna()
)

print("df_model shape:", df_model.shape)
print(df_model.head())

# ----------------------------
# 5) Format MLForecast (long + exog cols)
# ----------------------------
ts_lr = df_model.rename(columns={"date": "ds"})
ts_lr["unique_id"] = "UNRATE"

exog_cols = [c for c in ts_lr.columns if c not in ["unique_id", "ds", "y"]]
ts_lr = ts_lr[["unique_id", "ds", "y"] + exog_cols]

print("ts_lr shape:", ts_lr.shape)
print("Exog cols:", exog_cols[:8], "..." if len(exog_cols) > 8 else "")
ts_lr.head()

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
Nb dates: 801
Date min: 1959-01-01 00:00:00 | Date max: 2025-09-01 00:00:00
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
df_stationary shape: (8679, 3)
  series_id       date     value
0  BUSLOANS 1960-01-01  0.011578
1    INDPRO 1960-01-01  0.091976
2     USREC 1960-01-01  0.000000
3      M2SL 1960-01-01  0.001323
4  CPIAUCSL 1960-01-01 -0.006156
df_model shape: (777, 13)
         date    y  y_lag12  BUSLOANS_lag12  CPIAUCSL_lag12  \
12 1961-01-01  1.4     -0.8        0.011578       -0.006156   
13 1961-02-01  2.1     -1.1        0.011905       -0.003767   
14 1961-03-01  1.5     -0.2       -0.008356       -0.005455   
15 1961-04-01  

,unique_id,ds,y,y_lag12,BUSLOANS_lag12,CPIAUCSL_lag12,DPCERA3M086SBEA_lag12,INDPRO_lag12,M2SL_lag12,OILPRICEX_lag12,RPI_lag12,SP500_lag12,TB3MS_lag12,USREC_lag12
12,UNRATE,1961-01-01,1.4,-0.8,0.011578,-0.006156,0.001204,0.091976,0.001323,0.0,0.020977,0.017909,0.30,0.0
13,UNRATE,1961-02-01,2.1,-1.1,0.011905,-0.003767,0.006009,0.076960,0.002007,0.0,0.014565,-0.025663,-0.19,0.0
14,UNRATE,1961-03-01,1.5,-0.2,-0.008356,-0.005455,0.021240,0.007959,0.001324,0.0,0.006250,-0.070857,-1.18,0.0
15,UNRATE,1961-04-01,1.8,0.0,-0.009098,0.005090,0.033752,-0.025916,0.000634,0.0,0.006489,-0.040442,-1.12,0.0
16,UNRATE,1961-05-01,2.0,0.0,-0.000359,0.003383,0.009040,-0.018119,0.003977,0.0,0.007747,-0.010090,-0.67,1.0


# Dictionnaire de modèle

In [13]:
MLF_MODELS = {
    "LR_FULL_INFO_H12": lambda freq: MLForecast(
        models=[LinearRegression()],
        freq=freq,
    )
}

# Backtesting

In [16]:
# ============================================================
# Backtesting h=12 avec préproc (winsor + normalisation) NO-LEAKAGE
# Compatible versions MLForecast où PI se passent au FIT
# ============================================================

import numpy as np
import pandas as pd
from mlforecast.utils import PredictionIntervals


# ---------- Préproc ----------
def fit_preproc(X: pd.DataFrame, wins: float = 0.01, do_norm: bool = True):
    lower = X.quantile(wins)
    upper = X.quantile(1 - wins)
    Xw = X.clip(lower=lower, upper=upper, axis=1)

    if do_norm:
        mean = Xw.mean()
        std = Xw.std(ddof=0).replace(0, 1)
        Xn = (Xw - mean) / std
        prep = {"lower": lower, "upper": upper, "mean": mean, "std": std, "norm": True}
        return Xn, prep
    else:
        prep = {"lower": lower, "upper": upper, "mean": None, "std": None, "norm": False}
        return Xw, prep


def apply_preproc(X: pd.DataFrame, prep: dict):
    Xp = X.clip(lower=prep["lower"], upper=prep["upper"], axis=1)
    if prep.get("norm", False):
        std = prep["std"].replace(0, 1)
        Xp = (Xp - prep["mean"]) / std
    return Xp


def run_backtesting_h12_with_preproc(
    mlf,
    ts: pd.DataFrame,
    *,
    h: int = 12,
    step_size: int = 12,
    n_windows: int = 35,
    wins: float = 0.01,
    do_norm: bool = True,
    pi_windows: int = 3,
    levels: list = [95],
):
    # --- sanity checks
    required = {"unique_id", "ds", "y"}
    missing = required - set(ts.columns)
    if missing:
        raise ValueError(f"ts missing required columns: {missing}")

    ts = ts.copy()
    ts["ds"] = pd.to_datetime(ts["ds"])
    ts = ts.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    META_COLS = ["unique_id", "ds"]
    Y_COL = "y"
    EXOG_COLS = [c for c in ts.columns if c not in META_COLS + [Y_COL]]
    if not EXOG_COLS:
        raise ValueError("No exogenous columns found. EXOG_COLS is empty.")

    all_dates = np.array(sorted(ts["ds"].unique()))

    # Prediction intervals (Conformal)
    pi = PredictionIntervals(
        h=h,
        n_windows=pi_windows,
        method="conformal_distribution",
    )

    out = []

    for i in range(n_windows):
        cutoff_idx = len(all_dates) - (h + i * step_size) - 1
        if cutoff_idx < 0:
            break
        cutoff = all_dates[cutoff_idx]

        train = ts[ts["ds"] <= cutoff].copy()
        test = ts[(ts["ds"] > cutoff) & (ts["ds"] <= cutoff + pd.DateOffset(months=h))].copy()
        if test.empty:
            break

        # --- fit preproc sur TRAIN uniquement
        X_train = train[EXOG_COLS]
        X_test = test[EXOG_COLS]
        X_train_p, prep = fit_preproc(X_train, wins=wins, do_norm=do_norm)
        X_test_p = apply_preproc(X_test, prep)

        train_p = train.copy()
        test_p = test.copy()
        train_p[EXOG_COLS] = X_train_p
        test_p[EXOG_COLS] = X_test_p

        # --- FIT (✅ static_features=[] + ✅ prediction_intervals AU FIT)
        mlf.fit(
            train_p,
            static_features=[],
            prediction_intervals=pi,
        )

        # --- PREDICT (❌ pas de prediction_intervals ici dans ta version)
        # Selon les versions, level peut être requis ou non.
        # On le passe : si non supporté, enlève level=levels.
        fcst = mlf.predict(
            h=h,
            X_df=test_p,
            level=levels,
        )

        fcst["cutoff"] = cutoff
        out.append(fcst)

    if not out:
        raise RuntimeError("No backtesting windows were produced. Check dates/h/step_size/n_windows.")

    return pd.concat(out).reset_index(drop=True)

# Run 

In [17]:
# ============================================================
# RUN – Linear Regression (Nixtla MLForecast) | Full-information (h=12)
# ============================================================

from pathlib import Path

# ----------------------------
# Project root
# ----------------------------
PROJECT_ROOT = Path.cwd().parent
print("PROJECT_ROOT =", PROJECT_ROOT.resolve())

# ----------------------------
# CV / PI config
# ----------------------------
H = 12
STEP_SIZE = 12
PARTITIONS = 35
PI_WINDOWS = 3
LEVELS = [95]

# ----------------------------
# Model registry (must exist above)
# Example expected:
# MLF_MODELS = {
#     "LR_FULL_INFO_H12": lambda freq: MLForecast(
#         models=[LinearRegression()],
#         freq=freq,
#     )
# }
# ----------------------------
MODEL_KEY = "LR_FULL_INFO_H12"

print("Available models:", list(MLF_MODELS.keys()))
assert MODEL_KEY in MLF_MODELS, f"MODEL_KEY '{MODEL_KEY}' not found in MLF_MODELS"

# ----------------------------
# Instantiate model
# ----------------------------
mlf = MLF_MODELS[MODEL_KEY](FREQ)

# ✅ safe prints for MLForecast
model_names = list(mlf.models.keys()) if hasattr(mlf, "models") else []
first_name = next(iter(mlf.models)) if model_names else None

print("Running model key:", MODEL_KEY)
print("Running models:", model_names)
if first_name is not None:
    print("First model class:", mlf.models[first_name].__class__.__name__)
print("Freq:", FREQ)

# ----------------------------
# Sanity checks on data
# ----------------------------
assert {"unique_id", "ds", "y"}.issubset(ts_lr.columns), "ts_lr must contain unique_id, ds, y"
ts_lr = ts_lr.sort_values(["unique_id", "ds"]).reset_index(drop=True)

# (optional) quick check for full-information features
assert "y_lag12" in ts_lr.columns, "Expected 'y_lag12' in ts_lr (manual lag feature)"
assert any(c.endswith("_lag12") for c in ts_lr.columns if c not in ["unique_id", "ds", "y"]), \
    "Expected at least one exogenous *_lag12 column in ts_lr"

# ----------------------------
# Run backtesting (uses your function)
# IMPORTANT: your run_backtesting_h12_simple must pass static_features=[]
# ----------------------------
bkt_lr = run_backtesting_h12_with_preproc(
    mlf=mlf,
    ts=ts_lr,
    h=H,
    step_size=STEP_SIZE,
    n_windows=PARTITIONS,   # même valeur que ton PARTITIONS=35
    wins=0.01,
    do_norm=True,
    pi_windows=PI_WINDOWS,
    levels=LEVELS,
)

bkt_lr.head()

PROJECT_ROOT = D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA
Available models: ['LR_FULL_INFO_H12']
Running model key: LR_FULL_INFO_H12
Running models: ['LinearRegression']
First model class: LinearRegression
Freq: MS


,unique_id,ds,LinearRegression,LinearRegression-lo-95,LinearRegression-hi-95,cutoff
0,UNRATE,2024-10-01,-0.010712,-2.026479,2.005055,2024-09-01
1,UNRATE,2024-11-01,-0.222502,-1.571871,1.126866,2024-09-01
2,UNRATE,2024-12-01,-0.443575,-2.110205,1.223054,2024-09-01
3,UNRATE,2025-01-01,-0.305578,-1.278167,0.667011,2024-09-01
4,UNRATE,2025-02-01,-0.374318,-1.377978,0.629343,2024-09-01


In [19]:
# ============================================================
# (ADD) Build standardized OOS forecast table (Linear Regression) - ROBUST
# ============================================================

import numpy as np

# 0) Attacher y si absent (postRUN: predict() ne renvoie pas y)
if "y" not in bkt_lr.columns:
    bkt_lr = bkt_lr.merge(
        ts_lr[["unique_id", "ds", "y"]],
        on=["unique_id", "ds"],
        how="left",
        validate="many_to_one",
    )

# 1) Identifier colonne modèle (souvent "LinearRegression")
model_col = next((c for c in bkt_lr.columns if c.lower() == "linearregression"), None)
if model_col is None:
    raise KeyError(f"Model column not found. Available columns: {list(bkt_lr.columns)}")

# 2) Identifier lo/hi 95 (noms variables selon version)
lo_col = next(
    (c for c in bkt_lr.columns if c.lower() in {f"{model_col.lower()}-lo-95", f"{model_col.lower()}_lo_95", "linearregression-lo-95", "linearregression_lo_95"}),
    None
)
hi_col = next(
    (c for c in bkt_lr.columns if c.lower() in {f"{model_col.lower()}-hi-95", f"{model_col.lower()}_hi_95", "linearregression-hi-95", "linearregression_hi_95"}),
    None
)

# fallback pattern-match si pas trouvé exactement
if lo_col is None:
    lo_col = next((c for c in bkt_lr.columns if ("lo" in c.lower() and "95" in c.lower() and "linearregression" in c.lower())), None)
if hi_col is None:
    hi_col = next((c for c in bkt_lr.columns if ("hi" in c.lower() and "95" in c.lower() and "linearregression" in c.lower())), None)

# 3) Construire table standardisée
base_cols = ["unique_id", "ds", "cutoff", "y", model_col]
rename_map = {
    "unique_id": "series_id",
    "ds": "date",
    "y": "y_obs",
    model_col: "y_hat_lr",
}

if lo_col is not None and hi_col is not None:
    base_cols += [lo_col, hi_col]
    rename_map[lo_col] = "y_hat_lr_lo_95"
    rename_map[hi_col] = "y_hat_lr_hi_95"
else:
    print("⚠️ PI columns not found in bkt_lr (lo/hi). Output will be without intervals.")
    print("bkt_lr columns:", list(bkt_lr.columns))

df_lr_forecasts = (
    bkt_lr[base_cols]
    .rename(columns=rename_map)
    .sort_values(["series_id", "date"])
    .reset_index(drop=True)
)

print("Using model_col:", model_col)
print("Using lo_col:", lo_col, "| hi_col:", hi_col)
print("df_lr_forecasts shape:", df_lr_forecasts.shape)
df_lr_forecasts.head()

Using model_col: LinearRegression
Using lo_col: LinearRegression-lo-95 | hi_col: LinearRegression-hi-95
df_lr_forecasts shape: (420, 7)


,series_id,date,cutoff,y_obs,y_hat_lr,y_hat_lr_lo_95,y_hat_lr_hi_95
0,UNRATE,1990-10-01,1990-09-01,0.6,-0.265744,-1.503540,0.972053
1,UNRATE,1990-11-01,1990-09-01,0.8,0.095094,-1.471717,1.661905
2,UNRATE,1990-12-01,1990-09-01,0.9,-0.166198,-1.174062,0.841666
3,UNRATE,1991-01-01,1990-09-01,1.0,-0.073395,-0.760172,0.613381
4,UNRATE,1991-02-01,1990-09-01,1.3,-0.018788,-0.442367,0.404791


In [20]:
# ============================================================
# (ADD) 6) Save artifacts & outputs (Linear Regression)
# ============================================================

from datetime import datetime
import json

# ----------------------------
# Model identification
# ----------------------------
SERIES_ID = "UNRATE"
MODEL_TAG = "lr_lag12_exog"   # 🔑 clair et extensible

# ----------------------------
# Directories
# ----------------------------
OUTPUT_FORECASTS_DIR = PROJECT_ROOT / "outputs" / "forecasts"

ARTIFACT_CONFIGS_DIR = PROJECT_ROOT / "artifacts" / "configs"
ARTIFACT_CV_DIR      = PROJECT_ROOT / "artifacts" / "cv"
ARTIFACT_META_DIR    = PROJECT_ROOT / "artifacts" / "metadata"

OUTPUT_FORECASTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_CV_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_META_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# 1) Outputs (OOS forecasts)
# ----------------------------
oos_path = OUTPUT_FORECASTS_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_oos_forecasts.parquet"
df_lr_forecasts.to_parquet(oos_path, index=False)

# ----------------------------
# 2) Artifacts – raw backtesting output
# ----------------------------
bkt_path = ARTIFACT_CV_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_bkt_raw.parquet"
bkt_lr.to_parquet(bkt_path, index=False)

# ----------------------------
# 3) Run configuration (reproducibility)
# ----------------------------
run_config = {
    "model": "LinearRegression",
    "framework": "Nixtla-MLForecast",
    "target": SERIES_ID,
    "stationary": True,
    "lags": [12],
    "exogenous_variables": [
        c for c in ts_lr.columns if c not in ["unique_id", "ds", "y"]
    ],
    "horizon": H,
    "step_size": STEP_SIZE,
    "partitions": PARTITIONS,
    "prediction_intervals": {
        "method": "conformal_distribution",
        "levels": LEVELS,
        "n_windows": PI_WINDOWS,
    },
    "frequency": FREQ,
}

cfg_path = ARTIFACT_CONFIGS_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_config.json"
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

# ----------------------------
# 4) Metadata – run info
# ----------------------------
meta_path = ARTIFACT_META_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_run_info.json"
run_info = {
    "run_utc": datetime.utcnow().isoformat(),
    "project_root": str(PROJECT_ROOT.resolve()),
    "model_tag": MODEL_TAG,
    "files": {
        "oos_forecasts": str(oos_path.resolve()),
        "cv_raw": str(bkt_path.resolve()),
        "config": str(cfg_path.resolve()),
    },
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2)

print("✅ Saved:")
print(" - OOS forecasts :", oos_path.name)
print(" - CV raw        :", bkt_path.name)
print(" - Config        :", cfg_path.name)
print(" - Metadata      :", meta_path.name)


✅ Saved:
 - OOS forecasts : unrate_lr_lag12_exog_oos_forecasts.parquet
 - CV raw        : unrate_lr_lag12_exog_bkt_raw.parquet
 - Config        : unrate_lr_lag12_exog_config.json
 - Metadata      : unrate_lr_lag12_exog_run_info.json


C:\Users\Mita\AppData\Local\Temp\ipykernel_17756\677556810.py:72: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "run_utc": datetime.utcnow().isoformat(),


# Graphique

In [21]:
# ----------------------------
# Prepare obs
# ----------------------------
df_obs = (
    df_lr_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_obs": "y",
    })
    [["unique_id", "ds", "y"]]
)

In [22]:
# ----------------------------
# Prepare forecast + PI (LR)
# ----------------------------
df_fcst = (
    df_lr_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_hat_lr": "LR",
        "y_hat_lr_lo_95": "LR-lo-95",
        "y_hat_lr_hi_95": "LR-hi-95",
    })
    [[
        "unique_id",
        "ds",
        "LR",
        "LR-lo-95",
        "LR-hi-95",
    ]]
)

In [23]:
from utilsforecast.plotting import plot_series

fig = plot_series(
    df=df_obs,
    forecasts_df=df_fcst,
    level=[95],
    engine="plotly",
).update_layout(height=400)

# Rename legend entries
for trace in fig.data:
    if trace.name == "y":
        trace.name = "Unemployment rate (stationary)"
    elif trace.name == "LR":
        trace.name = "Linear Regression (lag-12 + exog)"
    elif "level_95" in trace.name.lower():
        trace.name = "95% Prediction Interval"

fig.show()

## Résultat
On voit notre modèle AR1 est très basique, il suit juste la direction du taux de chômage. Avec ce baseline, on s'aperçoit des informations très importantes pour orienter notre expérimentation. 

De 1990 à 2007, l'économie américaine a été stable. En 2008, elle a été frappée par la crise de Subprime. En 2019, çà été la crise de Coronavirus. Qu'est-ce qu'on peut dire de ces trois périodes? 

Globalement, le modèle auto-régressif reste proche des observations en période de stabilité. C'est une bonne capacité à capter la dynamique du chômage.

Lors des ruptures structurelles des deux crises, la qualité des prévisions se dégrade et les intervalles de prédiction s’élargissent. Ce qui traduit une incertitude de plus en plus élevée.

Le modèle capte la direction des variations, mais sa fiabilité diminue en période de crise, sans masquer cette incertitude.

Cette étude constitue un **sanity check du système de prévision**. Elle valide le comportement attendu du modèle et la cohérence du pipeline.

Une approche plus complexe est attendu. 

## Next
Essayons d'optimiser le paramètre "p" de AR pour confirmer notre analyse. 